In [ ]:
# Solved-1hop Aut-clean ladder @ 1M — HIGH_SPEEDUP — CONFIG
# Edit ONLY CHUNK_INDEX if needed (pre-set per file).
#
# Advisor gate (REVISE→addressed): freeze excludes bench60/66 + AC1M split +
# campaign_12h S-grid/arms + scale10k slices. Presentation-major job order.
# Mini plumbing verified @ budget 50 (not a ranking).
#
# Dataset = solved_1hop_autclean (~432 Aut-orbits, seed + moved_cov).
# HIGH_SPEEDUP = N_WORKERS="auto" + ENGINE="hcompact".
# On ~51 GB / 8-core Colab, memory-caps to ~6 workers at 1M (~7.6 GB/search).
#
# After the run: merge chunks, then
#   python3 -m experiments.heuristic_search.verify.verify_solved1hop_certs <jsonls>
# Report Δ vs baseline on complete rows only; stratify seed / moved_cov /
# short_relator. Never promote an arm from the budget-50 mini.

REPO_URL   = "https://github.com/Avi161/ACSolverX.git"
REPO_DIR   = "ACSolverX"
BRANCH     = "cursor/heur-12h-anti-overfit-a42e"
CLONE      = True
UPDATE_REPO = True

MOUNT_DRIVE = True
DRIVE_DIR   = "/content/drive/MyDrive/acsolverx/hsearch_solved1hop_1m"

CHUNK_INDEX = 2

cfg = dict(
    DATASET   = "solved_1hop_autclean",  # informational; wrapper loads freeze
    # SUBSET omitted — full chunk shard

    ARMS      = ['baseline', 's12', 's28', 's20_mk2', 's24_k1_mk2'],

    CHUNKS       = 5,
    CHUNK_INDEX  = CHUNK_INDEX,

    ENGINE       = "hcompact",   # REQUIRED @1M
    N_WORKERS    = "auto",
    KEEP_PATH    = True,

    NODE_BUDGET = 1_000_000,
    CHECKPOINTS = [1000, 5000, 10000, 25000, 50000, 100000, 250000, 500000, 1000000],
    MAX_RELATOR_LENGTH = 48,
    RESUME    = True,
    OUT_STEM  = "hsearch_solved1hop_1m",
    STAGE_DIR = "/content/hsearch_stage",
)

HEARTBEAT_SECS = 60
PROGRESS_SECS  = 300


In [ ]:
# ==================== SETUP (clone / pull / mount / purge) ================
import os, sys, subprocess, importlib

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numba numpy")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
    os.makedirs(cfg.get("STAGE_DIR", "/content/hsearch_stage"), exist_ok=True)
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
else:
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)
print("BRANCH:", BRANCH, "ENGINE:", cfg["ENGINE"], "N_WORKERS:", cfg["N_WORKERS"])
assert cfg["ENGINE"] == "hcompact", "ENGINE=hcompact required for HIGH_SPEEDUP @1M"

for _m in [m for m in sys.modules if m == "experiments" or m.startswith("experiments.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

from experiments.heuristic_search.core.hsolve import greedy_search_h
from experiments.heuristic_search.runners import run_solved1hop_scale as _rs
_ = greedy_search_h("xyx", "yx", 20, max_relator_length=32,
                    config=_rs.run_ab.ARMS["s20"])
print("kernels warm — setup done")
print("freeze rows:", len(_rs.load_frozen()), "dataset:", _rs.DATASET_NAME)
print("tip: Runtime → Restart → Run All resumes (UPDATE_REPO + flock + RESUME)")
print("ARMS:", cfg["ARMS"])


In [ ]:
# ==================== RUN (HIGH_SPEEDUP multi-worker) =====================
from experiments.heuristic_search.runners.run_solved1hop_scale import (
    run_solved1hop_scale, load_frozen)
from experiments.heuristic_search.runners import run_ab as _ra
nw, per = _ra._resolve_workers(
    {"N_WORKERS": cfg["N_WORKERS"]}, cfg["NODE_BUDGET"],
    cfg["MAX_RELATOR_LENGTH"], cfg["ENGINE"], cfg["KEEP_PATH"])
n_freeze = len(load_frozen())
print(f"HIGH_SPEEDUP resolve: N_WORKERS={cfg['N_WORKERS']} -> {nw} workers "
      f"(~{per:.1f} GB/search est., ENGINE={cfg['ENGINE']})")
print(f"budget={cfg['NODE_BUDGET']:,} chunk={cfg['CHUNK_INDEX']}/{cfg['CHUNKS']} "
      f"arms={cfg['ARMS']} freeze_n={n_freeze}")
print("job order: presentation-major (all arms per row before next row)")
run_solved1hop_scale(
    cfg,
    out_dir=(DRIVE_DIR if (IN_COLAB and MOUNT_DRIVE) else "results/hsearch"),
    heartbeat_secs=HEARTBEAT_SECS,
    progress_secs=PROGRESS_SECS,
)
print("done — leave session up until jsonl mirror finishes.")
print("next: merge chunks, then verify_solved1hop_certs on the merged jsonl.")
